# 강의 03 · 실습 1 — 에이전트 동작 원리 · (3.5) 디버깅

## 1. 문제상황

- 시립도서관 안내 데스크에는 「토요일에 몇 시까지 여나요」 같은 운영시간 질문과 「이 책 빌릴 수 있나요」 같은 소장·대출 질문이 함께 들어옵니다.
- 빌릴 수 있는 책이면 그 자리에서 회원번호로 예약까지 해 달라는 요청이 이어집니다.
- 담당자는 운영시간표를 보고, 도서 관리 화면에서 책을 찾고, 예약 화면에 회원번호를 넣는 세 가지 일을 질문에 따라 골라서 합니다.
- 언어 모델은 도서관의 운영시간표도 소장 목록도 예약 화면도 알지 못합니다.

## 2. 문제와 목표

- **문제**: 질문마다 운영시간표 조회·소장 조회·예약 등록 중 어느 일을 몇 번 해야 하는지 사람이 정하고 직접 합니다. 아래 「6. 코드 — 스텝바이스텝」의 완성 코드는 이 목표를 잘못 구현해, 문법 오류 없이 실행되지만 동작이 요구사항과 어긋나는 결함이 세 곳 있습니다. 세 곳을 모두 고쳐 「7. 실행 결과 확인」을 통과시키는 것이 과제입니다.
- **목표**
  - 질문을 입력하면 모델이 세 도구 중 필요한 도구를 골라 호출해야 합니다.
    - 세 도구: 운영시간표 조회, 소장 조회, 예약 등록
  - 앞 도구의 결과를 보고 다음 도구를 이어서 호출한 뒤 최종 답을 만드는 안내 프로그램을 만듭니다.
- **목표 달성 여부의 판정 기준**:
  - 운영시간 질문에서는 `opening_hours` 호출 한 번으로 끝나고,
  - 예약 요청에서는 `find_book` 호출 뒤 `reserve_book` 호출이 이어져 최종 답에 예약 완료가 들어 있는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex01_s3_diagram.svg)

## 4. 단계별 요구사항

1. **도구를 선언합니다.**
    - `opening_hours(day)`는 운영시간표 사전(평일·토요일·일요일 — 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다)에서 요일의 운영시간을 돌려주고, 없는 요일이면 「해당 요일의 정보가 없습니다」를 돌려줍니다.
    - `find_book(title)`은 소장 목록 사전에서 책의 소장 여부와 대출 가능 여부를 돌려주고, 없는 책이면 「소장하지 않는 책입니다」를 돌려줍니다.
    - `reserve_book(title, member_id)`는 예약을 등록하고 「<책 제목> 예약 완료 (회원 <회원번호>)」를 돌려줍니다.
    - 세 함수 모두 `@tool(parse_docstring=True)`를 붙이고, 독스트링에 도구 설명과 인자 설명을 적습니다.
    - 인자 설명에는 인자가 받는 값의 형식(요일 이름, 책 제목, 회원번호)을 함수가 실제로 받는 값과 같게 적습니다.
2. **도구를 모델에 묶습니다.**
    - 세 도구를 `bind_tools`로 묶어 `llm_tools`를 만들고, 세 도구 모두를 담은 `TOOLS` 사전을 만듭니다.
3. **모델을 호출하고 도구 호출 요청을 판정합니다.**
    - 사용자 질문을 `HumanMessage`로 담은 대화 기록을 `llm_tools`에 넣어 호출하고, 돌아온 응답의 `tool_calls`가 비어 있는지로 도구 호출 요청 여부를 판정합니다.
    - 「『파이썬 입문』이 대출 가능하면 회원번호 A123으로 예약해 주세요.」를 첫 질문으로 넣어 첫 호출의 `tool_calls`를 확인합니다.
4. **도구 결과를 되먹여 반복합니다.**
    - 도구 호출 요청이 있으면 모델 응답을 대화 기록에 붙이고, 요청된 도구를 `TOOLS`에서 찾아 인자로 실행한 뒤, 결과를 호출 id와 함께 `ToolMessage`로 대화 기록에 붙이고 모델을 다시 호출하는 `run_agent` 함수를 만듭니다.
    - 여기에 몇 번째 호출인지와 그 응답에 실린 도구 호출 요청의 수를 화면에 출력하는 줄을 더합니다.
    - 반복 상한은 4회이고, 상한에 닿으면 「반복 한도 초과」를 돌려줍니다.
5. **두 질문으로 실행합니다.**
    - 「토요일에는 몇 시까지 여나요?」와 「『파이썬 입문』이 대출 가능하면 회원번호 A123으로 예약해 주세요.」를 차례로 넣어, 호출 번호, 도구 이름과 인자, 도구 결과, 최종 답을 화면에 출력합니다.

## 5. 코드 골격 — 도구 호출 루프 4단

랭체인 문법으로 도구 호출 루프를 세우는 순서는 다음 네 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 네 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 도구 선언 | 파이썬 함수 위에 표시 한 줄을 붙여 도구로 만듭니다 | `@tool(parse_docstring=True)` | 1 |
| ② 도구 묶기 | 도구 목록을 모델에 붙여 도구를 쥔 모델을 만듭니다 | `llm.bind_tools([...])` | 2 |
| ③ 반복 호출·판정 | 대화 기록을 넣어 호출하고, 도구 호출이 실렸는지 봅니다 | `res.tool_calls` | 3 |
| ④ 결과 되먹임 | 도구를 실행하고 그 결과를 대화 기록에 붙여 다시 호출합니다 | `ToolMessage(content=..., tool_call_id=...)` | 4, 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from datetime import datetime
from dotenv import load_dotenv, find_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

### 단계 ① — 도구 선언 (요구사항 1)

- 도구의 실체는 파이썬 함수입니다. 함수 위에 `@tool`을 붙이면 함수 이름·독스트링·인자 타입에서 모델에게 보여 줄 도구 설명(스키마)이 만들어집니다.
- `parse_docstring=True`는 독스트링의 `Args:` 항목을 인자 설명으로 씁니다. 모델은 도구 설명과 인자 설명을 보고 어느 도구를 언제 부를지 정합니다.

In [ ]:
HOURS = {"평일": "09:00~20:00", "토요일": "09:00~17:00", "일요일": "휴관"}
BOOKS = {
    "파이썬 입문": "대출 가능 (3층 자연과학 서가)",
    "데이터 분석 첫걸음": "대출 중 (반납 예정일 9월 10일)",
    "인공지능 이야기": "대출 가능 (2층 교양 서가)",
}


@tool(parse_docstring=True)
def opening_hours(day: str) -> str:
    """시립도서관의 요일별 운영시간을 돌려준다. 몇 시까지 여는지 묻는 질문에 쓴다.

    Args:
        day: 조회할 날짜. 'YYYY-MM-DD' 형식으로 넘긴다. 예: 2026-09-05
    """
    return HOURS.get(day, "해당 요일의 정보가 없습니다.")


@tool(parse_docstring=True)
def find_book(title: str) -> str:
    """책 제목으로 소장 여부와 대출 가능 여부를 조회한다. 책을 빌릴 수 있는지 묻는 질문에 쓴다.

    Args:
        title: 책 제목. 겹낫표 없이 제목만 넘긴다.
    """
    return BOOKS.get(title, "소장하지 않는 책입니다.")


@tool(parse_docstring=True)
def reserve_book(title: str, member_id: str) -> str:
    """대출 가능한 책을 회원번호로 예약한다. 예약을 요청받았고 책이 대출 가능할 때 쓴다.

    Args:
        title: 책 제목.
        member_id: 회원번호. 예: A123
    """
    return f"{title} 예약 완료 (회원 {member_id})"


for t in [opening_hours, find_book, reserve_book]:
    print(f"[도구] {t.name}: {t.description} / 인자: {list(t.args)}")

### 단계 ② — 도구 묶기 (요구사항 2)

- `bind_tools`는 도구 목록을 모델에 붙여, 호출할 때마다 도구 설명을 함께 보내는 새 모델 객체를 돌려줍니다. 원래의 `llm`은 바뀌지 않습니다.
- `TOOLS`는 모델이 보낸 도구 이름을 실제 도구 객체로 바꾸는 사전입니다. 단계 ④에서 도구를 실행할 때 씁니다.

In [ ]:
llm_tools = llm.bind_tools([opening_hours, find_book, reserve_book])
TOOLS = {t.name: t for t in [opening_hours, find_book]}

print("모델에 묶인 도구:", list(TOOLS))

### 단계 ③ — 반복 호출·판정 (요구사항 3)

- 대화 기록은 메시지 객체의 리스트입니다. 첫 항목은 사용자 질문을 담은 `HumanMessage`입니다.
- 도구를 쥔 모델을 호출하면 `AIMessage`가 돌아옵니다. 모델이 도구를 부르기로 정했으면 `tool_calls`에 도구 이름·인자·호출 id가 실립니다. `tool_calls`가 비어 있으면 그 응답이 최종 답입니다.
- 아래 셀은 루프를 돌리기 전에 예약 요청의 첫 호출 응답을 그대로 열어 봅니다. 모델은 예약에 앞서 책을 먼저 찾아야 하므로 첫 응답에는 `find_book` 요청이 실립니다. 인자는 처음부터 딕셔너리로 돌아오므로 문자열 파싱이 필요 없습니다.

In [ ]:
messages = [HumanMessage(content="『파이썬 입문』이 대출 가능하면 회원번호 A123으로 예약해 주세요.")]
res = llm_tools.invoke(messages)

print("응답 종류:", type(res).__name__)
print("tool_calls:", res.tool_calls)
if res.tool_calls:
    print("판정: 도구 호출 요청이 있습니다. 도구 실행으로 갑니다.")
else:
    print("판정: 도구 호출 요청이 없습니다. 최종 답입니다.")

### 단계 ④ — 결과 되먹임 (요구사항 4, 5)

- 도구 호출 요청이 있으면 모델 응답(`AIMessage`)을 먼저 대화 기록에 붙입니다. 그 다음 요청된 도구를 `TOOLS`에서 찾아 `invoke(인자)`로 실행합니다.
- 도구 결과는 `ToolMessage`로 대화 기록에 붙입니다. `tool_call_id`는 어느 요청에 대한 결과인지 모델에게 알려 줍니다.
- 붙인 뒤 모델을 다시 호출합니다. 호출·판정·되먹임을 `for` 문 안에 넣으면 도구 호출 루프가 됩니다. 종료 조건은 도구 호출 요청이 없는 응답과 반복 한도 도달 둘입니다.
- 호출 번호와 도구 호출 요청의 수를 함께 출력하면, 질문마다 루프가 몇 바퀴 돌았는지 실행 기록에서 바로 읽을 수 있습니다.

In [ ]:
def run_agent(question: str, max_turn: int = 1) -> str:
    """질문을 받아 도구 호출 루프를 돌리고 최종 답을 돌려준다."""
    messages = [HumanMessage(content=question)]
    for turn in range(1, max_turn + 1):
        res = llm_tools.invoke(messages)
        print(f"  [{turn}번째 호출] 도구 호출 요청 {len(res.tool_calls)}개")
        if not res.tool_calls:
            return res.content
        messages.append(res)
        for call in res.tool_calls:
            print(f"    [도구 호출] {call['name']} {call['args']}")
            result = TOOLS[call["name"]].invoke(call["args"])
            print(f"    [도구 결과] {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    return "반복 한도 초과"

QUESTIONS = [
    "토요일에는 몇 시까지 여나요?",
    "『파이썬 입문』이 대출 가능하면 회원번호 A123으로 예약해 주세요.",
]

for i, q in enumerate(QUESTIONS, 1):
    print(f"=== {i}번 질문: {q} ===")
    answer = run_agent(q)
    print(f"  [최종 답] {answer}")
    print()

## 7. 실행 결과 확인

결함을 고친 뒤 다시 실행해 다음 세 가지를 확인합니다.

1. 1번 질문(토요일 운영시간)에서 `opening_hours {'day': '토요일'}`과 `[도구 결과] 09:00~17:00`이 찍히고 최종 답에 운영시간이 들어 있습니다.
2. 2번 질문(예약 요청)에서 1번째 호출에 `find_book`, 2번째 호출에 `reserve_book {'title': '파이썬 입문', 'member_id': 'A123'}`이 찍히고 최종 답에 「예약 완료」가 들어 있습니다.
3. 두 질문 모두 「반복 한도 초과」로 끝나지 않고, 실행 중에 오류로 멈추지 않습니다.

고치기 전에는 1번 질문의 도구 결과에 운영시간이 나오지 않고, 2번 질문이 「반복 한도 초과」로 끝납니다. 그 두 증상이 첫 실마리입니다.